# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [27]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN').strip()

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

Connected.


In [28]:
months = con.sql(f"""
    SELECT DISTINCT month
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    ORDER BY month DESC
    LIMIT 10
""").df()

print(months)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     month
0  2026-06
1  2026-05
2  2026-04
3  2026-03
4  2026-02
5  2026-01
6  2025-12
7  2025-11
8  2025-10
9  2025-09


In [29]:
files = con.sql("SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')").df()
print(files)

                                                                                                      file
0                                                hf://datasets/FlyRank/internship-warehouse/.gitattributes
1                                                     hf://datasets/FlyRank/internship-warehouse/README.md
2                                           hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
3                                           hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
8   hf://datasets/FlyRank/internship-

In [30]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

files = con.sql("SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')").df()
for f in files['file']:
    print(f)

hf://datasets/FlyRank/internship-warehouse/.gitattributes
hf://datasets/FlyRank/internship-warehouse/README.md
hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse

In [31]:
sample_df = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    LIMIT 5
""").df()

print(sample_df.columns.tolist())
sample_df

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis + time window

The unit of analysis is one anonymized content page, for one anonymized client, on one report date.

Source: fact_content_daily_performance_sample.parquet, representing the most recent full month available (June 2026).

Each row combines Google Search Console signals (impressions, clicks, average position) and Google Analytics signals (pageviews, sessions, engagement) for that page on that day, along with a breakdown of traffic sources including AI-assistant referrals (ChatGPT, Claude, Gemini, Perplexity, Copilot, Meta AI).

A verification query found 6,390 duplicate rows (about 0.05% of 11,694,072 total rows). The duplication follows a fixed pattern: the same ~355 client/content pairs are each duplicated on every day from June 13 through June 30, 2026 — not random noise, but a consistent data pipeline issue affecting a specific subset of pages. For this analysis, rows will be deduplicated (keeping one row per report_date + client + content combination) before any aggregation or feature-building.

In [32]:
# Confirm the grain: one row per report_date + client + content
check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_combinations
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

print(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_combinations
0    11694072             11687682


In [ ]:
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 10
""").df()

print(dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
dupe_summary = con.sql(f"""
    SELECT report_date, COUNT(*) AS dupe_rows
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
    )
    GROUP BY report_date
    ORDER BY dupe_rows DESC
""").df()

print(dupe_summary)

In [ ]:
which_pairs = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(DISTINCT report_date) AS days_duplicated
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
    )
    GROUP BY client_hash_id, content_hash_id
    ORDER BY days_duplicated DESC
    LIMIT 10
""").df()

print(which_pairs)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: feature / label / context / excluded

### Features
These describe observable page performance and will be used as model inputs:

- gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position (Google Search Console signals)
- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec (Google Analytics signals)
- sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai (traffic source breakdown)
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other (AI-assistant referral breakdown)
- scroll_events

### Label
This table does not contain a pre-built outcome label. A proxy label (opportunity_proxy) will be engineered later, once trend direction is calculated across multiple days of this daily data.

### Context
Useful for grouping, joining, and understanding the data, but not fed directly into the model:

- report_date, month (time context)
- client_hash_id, content_hash_id (row identifiers)
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available (data coverage flags)

### Excluded

- client_hash_id — excluded as a model feature (kept only as context/for grouping). Using it as a feature could let the model memorize specific clients instead of learning generalizable content-performance patterns.
- content_hash_id — same reasoning; excluded as a feature, kept only as a row identifier.

In [ ]:
coverage = con.sql(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS row_count
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY row_count DESC
""").df()

print(coverage)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Data coverage verification

Only about 5% of rows have both GSC and GA4 data available; roughly 54% have neither. This is a major consideration for feature selection and will directly shape what the model can reliably learn from.

In [ ]:
# Data coverage check — how much of the data has usable signals?
coverage = con.sql(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY row_count DESC
""").df()

print(coverage)

In [ ]:
missing_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS missing_sessions
FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

missing_check

In [ ]:
history_check = con.sql(f"""
    SELECT gsc_data_start, ga4_data_start, COUNT(*) AS n_clients
    FROM read_parquet('{rel}/dim_clients.parquet')
    GROUP BY gsc_data_start, ga4_data_start
    ORDER BY n_clients DESC
    LIMIT 10
""").df()

print(history_check)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can show observed relationships between content performance signals, but it cannot prove that changing a page will cause better rankings or traffic.

The dataset also has uneven data availability. Around 54% of rows have neither GSC nor GA4 data available, meaning some pages have limited measurable signals.

The daily records may contain overlapping time windows or duplicated pipeline records, so deduplication and careful aggregation are required before feature engineering.

The data does not contain a true business outcome label (such as confirmed successful content refreshes), so any prediction target will need to be treated as a proxy for decision support.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.